# Instrument-Agnostic Automatic Music Transcription (Colab Inference)

This notebook allows you to run inference using the [Instrument-Agnostic AMT](https://github.com/anime-song/instrument-agnostic-amt) model.
It will automatically download the pre-trained model from Hugging Face.

## 1. Setup Environment

In [ ]:
# @title Install dependencies
!git clone https://github.com/anime-song/instrument-agnostic-amt.git
%cd instrument-agnostic-amt
!uv pip install -r requirements.txt
!uv pip install yt-dlp stem-splitter # Optional: yt-dlp for downloading from YouTube


## 2. Prepare Audio

You can either upload a file or download one from a URL (e.g., YouTube).

In [ ]:
# @title Upload audio file
from google.colab import files
import os

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

## 3. Run Inference

In [ ]:
# @title Run Transcription
!python infer.py --audio "{audio_path}"

import os
midi_path = os.path.splitext(audio_path)[0] + ".mid"
if os.path.exists(midi_path):
    print(f"Success! MIDI saved to: {midi_path}")
else:
    print("Error: MIDI file was not generated.")

## 4. Download Results

In [ ]:
# @title Download MIDI file
if os.path.exists(midi_path):
    files.download(midi_path)
else:
    print("No MIDI file to download.")

## 5. Optional: Stem Separation -> Transcribe Each Stem -> Merge

This section mirrors the `batch_process_unlabeled.py` flow inside Colab.
It separates the uploaded song into stems, transcribes each non-drum stem, and merges the per-stem MIDI files into one result.


In [ ]:
# @title Prepare stem-separated transcription helpers
from collections import defaultdict
from pathlib import Path
import shutil

import librosa
import numpy as np
import pretty_midi
import soundfile as sf
import torch

import infer
from stem_splitter.inference import SeparationConfig, _separate_one_file, load_mss_model


# セッション中にモデルを使い回して、再実行時の待ち時間を減らす。
STEM_PIPELINE_CACHE = {}


def merge_midis_logic(midi_paths, output_file, max_melodic=15):
    """ステムごとの MIDI を 1 本にまとめる。"""
    if not midi_paths:
        raise ValueError("No MIDI files to merge")

    # 1. 先頭 MIDI のテンポマップを土台として使う。
    master_pm = pretty_midi.PrettyMIDI(str(midi_paths[0]))

    all_notes = defaultdict(list)
    all_ccs = defaultdict(list)
    all_pbends = defaultdict(list)
    instrument_names = {}

    # 2. 同じ program / drum 属性ごとにノートを集約する。
    for path in midi_paths:
        pm = pretty_midi.PrettyMIDI(str(path))
        for inst in pm.instruments:
            key = (inst.program, inst.is_drum, inst.name)
            filtered_notes = [n for n in inst.notes if (n.end - n.start) < 15.0]
            all_notes[key].extend(filtered_notes)
            all_ccs[key].extend(inst.control_changes)
            all_pbends[key].extend(inst.pitch_bends)
            if key not in instrument_names:
                instrument_names[key] = inst.name

    melodic_keys = [k for k in all_notes.keys() if not k[1]]
    drum_keys = [k for k in all_notes.keys() if k[1]]
    melodic_keys.sort(key=lambda k: len(all_notes[k]), reverse=True)

    final_instruments = []
    if len(melodic_keys) > max_melodic:
        kept_keys = melodic_keys[: max_melodic - 1]
        overflow_keys = melodic_keys[max_melodic - 1 :]
        for key in kept_keys:
            inst = pretty_midi.Instrument(
                program=key[0],
                is_drum=key[1],
                name=instrument_names[key],
            )
            inst.notes = all_notes[key]
            inst.control_changes = all_ccs[key]
            inst.pitch_bends = all_pbends[key]
            final_instruments.append(inst)

        base_key = overflow_keys[0]
        overflow_inst = pretty_midi.Instrument(
            program=base_key[0],
            is_drum=base_key[1],
            name="Other / Merged",
        )
        for key in overflow_keys:
            overflow_inst.notes.extend(all_notes[key])
            overflow_inst.control_changes.extend(all_ccs[key])
            overflow_inst.pitch_bends.extend(all_pbends[key])
        final_instruments.append(overflow_inst)
    else:
        for key in melodic_keys:
            inst = pretty_midi.Instrument(
                program=key[0],
                is_drum=key[1],
                name=instrument_names[key],
            )
            inst.notes = all_notes[key]
            inst.control_changes = all_ccs[key]
            inst.pitch_bends = all_pbends[key]
            final_instruments.append(inst)

    for key in drum_keys:
        inst = pretty_midi.Instrument(
            program=key[0],
            is_drum=key[1],
            name=instrument_names[key],
        )
        inst.notes = all_notes[key]
        inst.control_changes = all_ccs[key]
        inst.pitch_bends = all_pbends[key]
        final_instruments.append(inst)

    master_pm.instruments = final_instruments
    for inst in master_pm.instruments:
        inst.notes.sort(key=lambda note: note.start)
        inst.control_changes.sort(key=lambda x: x.time)
        inst.pitch_bends.sort(key=lambda x: x.time)
    master_pm.write(str(output_file))


def resolve_stem_paths(*, song_path, stem_dir, stem_names):
    """既存 stem 出力の標準パスを組み立て、存在するものだけ返す。"""
    song_file = Path(song_path)
    song_id = song_file.stem
    stem_root = Path(stem_dir) / song_id
    resolved = {}

    # 1. 標準の保存規則から期待パスを直接構成する。
    for stem_name in stem_names:
        expected_path = stem_root / f"{song_id}_{stem_name}.wav"
        if expected_path.exists():
            resolved[stem_name] = expected_path

    if resolved:
        return resolved

    # 2. 名前が少しずれていても拾えるよう、末尾 stem 名で緩く補完する。
    if not stem_root.exists():
        return resolved

    for wav_path in sorted(stem_root.glob("*.wav")):
        stem_key = wav_path.stem
        for stem_name in stem_names:
            if stem_key.endswith(f"_{stem_name}"):
                resolved.setdefault(stem_name, wav_path)
                break

    return resolved


def prepare_audio_for_stem_separation(audio_path, *, temp_dir):
    """Stem Separation 向けに入力音声を一時的に 2ch WAV へそろえる。"""
    audio_file = Path(audio_path)
    waveform, sample_rate = librosa.load(str(audio_file), sr=None, mono=False)

    if waveform.ndim == 1:
        waveform = waveform[None, :]
    elif waveform.ndim == 2 and waveform.shape[0] > waveform.shape[1]:
        waveform = waveform.T

    source_channels = int(waveform.shape[0])
    if source_channels <= 0:
        raise ValueError(f"Audio file has no channels: {audio_file}")
    if source_channels == 2:
        return audio_file

    # 1. mono は疑似ステレオ化し、2. 3ch 以上は先頭 2ch を使う。
    if source_channels == 1:
        waveform = np.repeat(waveform, 2, axis=0)
        channel_mode = "pseudo-stereo"
    else:
        waveform = waveform[:2]
        channel_mode = "first-two-channels"

    temp_dir = Path(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)
    prepared_path = temp_dir / f"{audio_file.stem}.wav"
    sf.write(str(prepared_path), waveform.T, samplerate=int(sample_rate))
    print(
        f"Prepared {channel_mode} input for stem separation: "
        f"source_channels={source_channels} -> {prepared_path.name}"
    )
    return prepared_path


def get_stem_pipeline_models(checkpoint_path=None, device_preference=None, model_type="default"):
    """AMT とステム分離モデルを読み込み、セッション中は再利用する。"""
    device = torch.device(device_preference or ("cuda" if torch.cuda.is_available() else "cpu"))
    checkpoint = infer._ensure_checkpoint(
        None if checkpoint_path in (None, "", "DEFAULT") else Path(checkpoint_path),
        model_type=model_type
    )
    amt_cache_key = ("amt", str(checkpoint.resolve()), device.type)
    sep_cache_key = ("sep", device.type)

    if sep_cache_key not in STEM_PIPELINE_CACHE:
        print(f"Loading Separation model on {device} ...")
        sep_config = SeparationConfig(skip_existing=True)
        sep_model = load_mss_model(sep_config, device=device)
        sep_dtype = torch.float16 if sep_config.use_half_precision and device.type == "cuda" else torch.float32
        STEM_PIPELINE_CACHE[sep_cache_key] = (sep_config, sep_model, sep_dtype)
    else:
        sep_config, sep_model, sep_dtype = STEM_PIPELINE_CACHE[sep_cache_key]

    if amt_cache_key not in STEM_PIPELINE_CACHE:
        print(f"Loading AMT model ({model_type}) on {device} ...")
        amt_model, amt_config, amt_settings = infer._load_model_and_settings(
            checkpoint,
            device=device,
            window_ms_override=None,
            stride_ms_override=None,
            track_batch_size_override=None,
        )
        STEM_PIPELINE_CACHE[amt_cache_key] = (amt_model, amt_config, amt_settings)
    else:
        print(f"Reusing cached AMT model ({model_type}) on {device} ...")
        amt_model, amt_config, amt_settings = STEM_PIPELINE_CACHE[amt_cache_key]

    return {
        "device": device,
        "checkpoint": checkpoint,
        "amt_model": amt_model,
        "amt_config": amt_config,
        "amt_settings": amt_settings,
        "sep_config": sep_config,
        "sep_model": sep_model,
        "sep_dtype": sep_dtype,
    }


def run_stem_separated_transcription(
    audio_path,
    *,
    checkpoint_path=None,
    output_root="colab_outputs",
    window_batch_size=4,
    max_midi_melodic_instruments=15,
    skip_drum_stems=True,
    cleanup_separated_stems=False,
    merge_onset_ms=20.0,
):
    """ステム分離 -> 各ステム採譜 -> MIDI マージを 1 回で実行する。"""
    audio_file = Path(audio_path)
    if not audio_file.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_file}")

    bundle = get_stem_pipeline_models(checkpoint_path=checkpoint_path, model_type="default")
    device = bundle["device"]
    amt_model = bundle["amt_model"]
    amt_config = bundle["amt_config"]
    amt_settings = bundle["amt_settings"]
    sep_config = bundle["sep_config"]
    sep_model = bundle["sep_model"]
    sep_dtype = bundle["sep_dtype"]

    bass_bundle = get_stem_pipeline_models(checkpoint_path=checkpoint_path, model_type="bass")
    bass_amt_model = bass_bundle["amt_model"]
    bass_amt_config = bass_bundle["amt_config"]
    bass_amt_settings = bass_bundle["amt_settings"]

    vocal_bundle = get_stem_pipeline_models(checkpoint_path=checkpoint_path, model_type="vocal_harmony")
    vocal_amt_model = vocal_bundle["amt_model"]
    vocal_amt_config = vocal_bundle["amt_config"]
    vocal_amt_settings = vocal_bundle["amt_settings"]

    guitar_bundle = get_stem_pipeline_models(checkpoint_path=checkpoint_path, model_type="guitar")
    guitar_amt_model = guitar_bundle["amt_model"]
    guitar_amt_config = guitar_bundle["amt_config"]
    guitar_amt_settings = guitar_bundle["amt_settings"]

    # 1. 出力先を曲ごとに分ける。
    run_root = Path(output_root) / audio_file.stem
    stem_dir = run_root / "stems"
    stem_midi_dir = run_root / "stem_midis"
    merged_dir = run_root / "merged"
    for directory in (stem_dir, stem_midi_dir, merged_dir):
        directory.mkdir(parents=True, exist_ok=True)

    # 2. まず元音源をステム分離する。
    separation_input = prepare_audio_for_stem_separation(
        audio_file,
        temp_dir=run_root / "prepared_inputs",
    )
    print(f"Separating stems for: {audio_file.name}")
    stems = _separate_one_file(
        separation_input,
        stem_dir,
        sep_config,
        sep_model,
        device,
        sep_dtype,
    )

    # 3. 再実行時に分離が省略された場合は、既存 stem の標準パスを復元する。
    if not stems:
        stems = resolve_stem_paths(
            song_path=audio_file,
            stem_dir=stem_dir,
            stem_names=sep_config.stem_names,
        )
        if stems:
            print(f"Reusing existing stems: {sorted(stems)}")
        else:
            raise RuntimeError(f"No stems found for {audio_file.stem}")

    # 4. ドラム以外の各ステムを順に採譜する。
    song_midi_paths = []
    for stem_name, stem_path in sorted(stems.items()):
        if skip_drum_stems and "drum" in stem_name.lower():
            print(f"Skipping drum stem: {stem_name}")
            continue

        output_midi = stem_midi_dir / f"{audio_file.stem}_{stem_name}.mid"
        print(f"Transcribing stem: {stem_name}")

        if "bass" in stem_name.lower():
            current_amt_model = bass_amt_model
            current_amt_config = bass_amt_config
            current_amt_settings = bass_amt_settings
        elif "vocal" in stem_name.lower():
            current_amt_model = vocal_amt_model
            current_amt_config = vocal_amt_config
            current_amt_settings = vocal_amt_settings
        elif "guitar" in stem_name.lower():
            current_amt_model = guitar_amt_model
            current_amt_config = guitar_amt_config
            current_amt_settings = guitar_amt_settings
        else:
            current_amt_model = amt_model
            current_amt_config = amt_config
            current_amt_settings = amt_settings

        waveform, _, _ = infer._load_audio(
            Path(stem_path),
            target_sample_rate=current_amt_config.sample_rate,
        )
        notes, _, _ = infer.run_inference(
            model=current_amt_model,
            waveform=waveform.to(device),
            model_config=current_amt_config,
            settings=current_amt_settings,
            device=device,
            amp_enabled=False,
            amp_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            velocity=100,
            merge_gap_ms=None,
            merge_onset_ms=merge_onset_ms,
            silence_gate_rms_dbfs=-72,
            window_batch_size=window_batch_size,
            max_midi_melodic_instruments=max_midi_melodic_instruments,
            disable_tqdm=True,
            max_note_seconds=15.0
        )

        midi = infer._build_midi(notes, sample_rate=current_amt_config.sample_rate, instrument_volumes=dict(infer.DEFAULT_INSTRUMENT_VOLUMES))
        midi.write(str(output_midi))
        song_midi_paths.append(output_midi)

    if not song_midi_paths:
        raise RuntimeError("No stem MIDI files were generated")

    # 5. ステムごとの MIDI を 1 本にまとめる。
    merged_midi_path = merged_dir / f"{audio_file.stem}.mid"
    merge_midis_logic(
        song_midi_paths,
        merged_midi_path,
        max_melodic=max_midi_melodic_instruments,
    )

    # 6. 必要なら中間の分離 wav を消して容量を節約する。
    if cleanup_separated_stems:
        shutil.rmtree(stem_dir, ignore_errors=True)

    result = {
        "audio_path": str(audio_file),
        "output_root": str(run_root),
        "stem_dir": str(stem_dir),
        "stem_midi_dir": str(stem_midi_dir),
        "merged_midi_path": str(merged_midi_path),
        "stem_count": len(stems),
        "transcribed_stem_count": len(song_midi_paths),
    }
    print("Merged MIDI:", merged_midi_path)
    return result


In [ ]:
# @title Run stem-separated transcription
OUTPUT_ROOT = "colab_outputs"  # @param {type:"string"}
WINDOW_BATCH_SIZE = 4  # @param {type:"integer"}
MAX_MIDI_MELODIC_INSTRUMENTS = 15  # @param {type:"integer"}
SKIP_DRUM_STEMS = True  # @param {type:"boolean"}
CLEANUP_SEPARATED_STEMS = False  # @param {type:"boolean"}
MERGE_ONSET_MS = 20.0  # @param {type:"number"}

if "audio_path" not in globals():
    raise RuntimeError("Please upload an audio file first.")

stem_pipeline_result = run_stem_separated_transcription(
    audio_path,
    checkpoint_path=None,
    output_root=OUTPUT_ROOT,
    window_batch_size=WINDOW_BATCH_SIZE,
    max_midi_melodic_instruments=MAX_MIDI_MELODIC_INSTRUMENTS,
    skip_drum_stems=SKIP_DRUM_STEMS,
    cleanup_separated_stems=CLEANUP_SEPARATED_STEMS,
    merge_onset_ms=MERGE_ONSET_MS,
)
stem_pipeline_result


In [ ]:
# @title Download stem-separated results
from google.colab import files
from pathlib import Path
import shutil

if "stem_pipeline_result" not in globals():
    print("Run the stem-separated transcription cell first.")
else:
    merged_midi_path = Path(stem_pipeline_result["merged_midi_path"])
    stem_midi_dir = Path(stem_pipeline_result["stem_midi_dir"])
    zip_base = stem_midi_dir.parent / f"{stem_midi_dir.name}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=stem_midi_dir))

    print(f"Downloading merged MIDI: {merged_midi_path}")
    files.download(str(merged_midi_path))
